# Eigen-Interactions Filtering

Apply the eigen-interactions package to sequences from the joint library (~57k).

### Steps

1. **Load sequences** from joint library

2. **Compute DeepLIFT/SHAP attributions** $A(s, k)$ for sequence $s$ across $k$ cell lines

3. **Compute Covariance matrix** $\mathbf{C}$ where:

$\displaystyle C_{ij} = \text{Cov}\!\big(A(s, k_i),\; A(s, k_j)\big) = \frac{1}{L}\sum_{\ell=1}^{L} A_\ell(s, k_i)\, A_\ell(s, k_j) \;-\; \bar{A}(s, k_i)\,\bar{A}(s, k_j)$

   - **Diagonal:** $C_{ii} = \text{Var}\!\big(A(s, k_i)\big)$
   - **Off-diagonal:** $C_{ij} = \text{Cov}\!\big(A(s, k_i),\; A(s, k_j)\big),\quad i \neq j$

4. **Find eigen-interactions** by diagonalizing $\mathbf{C}$:

$\displaystyle \mathbf{C} = \mathbf{V}\,\boldsymbol{\Lambda}\,\mathbf{V}^\top \qquad\Longrightarrow\qquad \text{EI}_m(s) = \sum_{k} V_{km}\, A(s, k)$

   so that in the new basis $(\text{EI}_1, \text{EI}_2, \ldots)$ the off-diagonals $\approx 0$.

5. **Plot** all $\text{EI}_1(s)$ projected onto 3D coordinates $(\text{HepG2},\; \text{K562},\; \text{WTC11})$.

In [13]:
import os, sys, importlib
import numpy as np
import pandas as pd

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

REPO = '/grid/wsbs/home_norepl/pmantill/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra'
EIGEN_DIR = os.path.join(REPO, 'eigen-interactions')
sys.path.insert(0, EIGEN_DIR)

import eigen_steering; importlib.reload(eigen_steering)
from eigen_steering import EigenMap, PROMOTER_SEQ, RAND_BARCODE

# Override paths to point at our local models instead of the training repo
eigen_steering.WEIGHTS_PATH = '/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/alphagenome_torch_MPRAMoCon/weights/model_fold_0.safetensors'
eigen_steering.RESULTS_DIR = os.path.join(REPO, 'models')

# Our model dirs have best_stage2.pt directly (no checkpoints/ subdir),
# so patch _load_model to look in the right place
_orig_load = EigenMap._load_model
def _patched_load(self, ct, squeeze=False):
    model_name = self.model_names[ct]
    ckpt_dir = os.path.join(eigen_steering.RESULTS_DIR, model_name)
    # Check both layouts: direct and checkpoints/
    direct = os.path.join(ckpt_dir, 'best_stage2.pt')
    nested = os.path.join(ckpt_dir, 'checkpoints', 'best_stage2.pt')
    if os.path.exists(direct) and not os.path.exists(nested):
        os.makedirs(os.path.join(ckpt_dir, 'checkpoints'), exist_ok=True)
        os.symlink(direct, nested)
    return _orig_load(self, ct, squeeze=squeeze)
EigenMap._load_model = _patched_load

DATA_CSV = os.path.join(REPO, 'data', 'joint_library_combined.csv')
print(f'Eigen-interactions loaded from {EIGEN_DIR}')

Eigen-interactions loaded from /grid/wsbs/home_norepl/pmantill/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra/eigen-interactions


In [14]:
# Load joint library
df = pd.read_csv(DATA_CSV)
df = df.dropna(subset=['sequence', 'K562_log2FC', 'HepG2_log2FC', 'WTC11_log2FC']).reset_index(drop=True)
print(f'{len(df)} sequences with K562 + HepG2 + WTC11 data')

# Best models per cell line by Pearson R on joint library (validate_models.ipynb)
# K562:  do075 → r=0.8915 | do06 → r=0.8837 | do03 → r=0.8683
# HepG2: do03  → r=0.8750 | do075 → r=0.8703 | do06 → r=0.8665
# WTC11: do075 → r=0.8457 | do06 → r=0.8371 | do03 → r=0.8262
MODEL_NAMES = {
    'K562':  'K562_v6_do075',
    'HepG2': 'HepG2_v6_do03',
    'WTC11': 'WTC11_v6_do075',
}

56974 sequences with K562 + HepG2 + WTC11 data


In [15]:
# Load precomputed DeepLIFT/SHAP attributions
# (computed via: bash submit_attributions.sh && bash submit_attributions.sh merge)
import glob

ATTR_PATH = os.path.join(REPO, 'genomic_targets', 'data', 'deeplift_attributions.npz')
SHARD_DIR = os.path.join(REPO, 'genomic_targets', 'data', 'attr_shards')

em = EigenMap(model_names=MODEL_NAMES, device='cuda')
em.load_from_dataframe(df, seq_col='sequence')
em.set_actual({ct: df[f'{ct}_log2FC'].values for ct in MODEL_NAMES})

if os.path.exists(ATTR_PATH):
    em.load_attributions(ATTR_PATH)
    print(f'\nAttribution shapes:')
    for ct in MODEL_NAMES:
        print(f'  {ct}: attr {em.attr[ct].shape}  attr_hyp {em.attr_hyp[ct].shape}')
else:
    print(f'Merged file not found: {ATTR_PATH}')
    print(f'Run: bash submit_attributions.sh merge')
    print(f'\nShard status:')
    for ct in MODEL_NAMES:
        shards = sorted(glob.glob(os.path.join(SHARD_DIR, f'{ct}_shard_*.npz')))
        # Only count shards that aren't being written (size > 0 and not locked)
        complete = []
        for s in shards:
            try:
                sz = os.path.getsize(s)
                if sz > 0:
                    np.load(s, allow_pickle=False)  # verify readable
                    complete.append(s)
            except Exception:
                pass
        print(f'  {ct}: {len(complete)}/10 shards complete')

EigenMap: ['K562', 'HepG2', 'WTC11'], models={'K562': 'K562_v6_do075', 'HepG2': 'HepG2_v6_do03', 'WTC11': 'WTC11_v6_do075'}
Loaded 56974 sequences, X shape: torch.Size([56974, 4, 281])
Merged file not found: /grid/wsbs/home_norepl/pmantill/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra/genomic_targets/data/deeplift_attributions.npz
Run: bash submit_attributions.sh merge

Shard status:
  K562: 0/10 shards complete
  HepG2: 0/10 shards complete
  WTC11: 0/10 shards complete
